# gnomAD Variant Frequencies

This notebook retrieves gnomAD variant frequency data for selected gene regions and classifies variants by allele-frequency thresholds.

In [1]:
import requests
import pandas as pd
import time
import random
import os
from pathlib import Path

In [2]:
gene_symbols_file = Path("../data/raw/gene_list/macrophage_gene_symbols.csv")
regions_file = Path("../data/interim/ensembl/gene_regions_grch38.csv")

out_dir = Path("../data/final")
out_dir.mkdir(parents=True, exist_ok=True)

out_csv = out_dir / "gnomad_frequencies.csv"

gnomad_api_url = "https://gnomad.broadinstitute.org/api"

dataset = "gnomad_r4"
reference_genome = "GRCh38"

In [3]:
gene_symbols = pd.read_csv(gene_symbols_file)

if "gene_symbol" not in gene_symbols.columns:
    raise ValueError(
        f"Expected column 'gene_symbol' was not found in {gene_symbols_file}. "
        f"Available columns are: {gene_symbols.columns.tolist()}"
    )

gene_symbols["gene_symbol"] = (
    gene_symbols["gene_symbol"]
    .astype(str)
    .str.strip()
)

gene_symbols = (
    gene_symbols
    .loc[gene_symbols["gene_symbol"] != ""]
    .drop_duplicates()
    .sort_values("gene_symbol")
    .reset_index(drop=True)
)

regions = pd.read_csv(regions_file, sep=";")

gene_regions = regions.loc[
    regions["input_gene_symbol"].str.upper().isin(
        gene_symbols["gene_symbol"].str.upper()
    )
].copy()

gene_regions = gene_regions.loc[
    gene_regions["lookup_status"] == "found"
].reset_index(drop=True)

missing_genes = sorted(
    set(gene_symbols["gene_symbol"].str.upper())
    - set(gene_regions["input_gene_symbol"].str.upper())
)

if missing_genes:
    print(f"Genes not found in {regions_file}: {missing_genes}")

gene_regions[[
    "input_gene_symbol",
    "official_gene_symbol",
    "chromosome",
    "gene_start",
    "gene_end",
    "assembly_name",
    "lookup_status",
]].head()

,input_gene_symbol,official_gene_symbol,chromosome,gene_start,gene_end,assembly_name,lookup_status
0,APOE,APOE,19,44903787,44909396,GRCh38,found
1,B2M,B2M,15,44711253,44718851,GRCh38,found
2,C1QA,C1QA,1,22635057,22640201,GRCh38,found
3,C1QB,C1QB,1,22652713,22661637,GRCh38,found
4,C1QC,C1QC,1,22642558,22648122,GRCh38,found


In [4]:
def make_region_chunks(start, end, chunk_size=25_000):
    chunks = []

    current_start = start

    while current_start <= end:
        current_end = min(current_start + chunk_size - 1, end)
        chunks.append((current_start, current_end))
        current_start = current_end + 1

    return chunks

In [5]:
gnomad_region_query = """
query RegionVariants(
  $chrom: String!,
  $start: Int!,
  $stop: Int!,
  $referenceGenome: ReferenceGenomeId!,
  $dataset: DatasetId!
) {
  region(
    chrom: $chrom,
    start: $start,
    stop: $stop,
    reference_genome: $referenceGenome
  ) {
    variants(dataset: $dataset) {
      variant_id
      rsids
      chrom
      pos
      ref
      alt
      consequence

      exome {
        ac
        an
        af
        populations {
          id
          ac
          an
        }
      }

      genome {
        ac
        an
        af
        populations {
          id
          ac
          an
        }
      }
    }
  }
}
"""

In [6]:
def query_gnomad_region(chromosome, start, stop, dataset="gnomad_r4",
                         max_retries=5, base_delay=5):
    variables = {
        "chrom": str(chromosome),
        "start": int(start),
        "stop": int(stop),
        "referenceGenome": reference_genome,
        "dataset": dataset,
    }

    for attempt in range(max_retries):
        try:
            response = requests.post(
                gnomad_api_url,
                json={
                    "query": gnomad_region_query,
                    "variables": variables,
                },
                headers={"Content-Type": "application/json"},
                timeout=60,
            )

            if response.status_code == 429:
                # Exponential backoff with jitter
                wait = base_delay * (2 ** attempt) + random.uniform(0, 2)
                print(f"  Rate limited (429). Waiting {wait:.1f}s before retry "
                      f"(attempt {attempt + 1}/{max_retries})...")
                time.sleep(wait)
                continue

            if not response.ok:
                print("Status code:", response.status_code)
                print("Response text:")
                print(response.text[:2000])
                response.raise_for_status()

            result = response.json()

            if "errors" in result:
                raise RuntimeError(result["errors"])

            region_data = result.get("data", {}).get("region")

            if region_data is None:
                return []

            return region_data.get("variants", [])

        except requests.exceptions.Timeout:
            wait = base_delay * (2 ** attempt)
            print(f"  Request timed out. Waiting {wait:.1f}s before retry "
                  f"(attempt {attempt + 1}/{max_retries})...")
            time.sleep(wait)

    raise RuntimeError(
        f"gnomAD query failed after {max_retries} retries for "
        f"chr{chromosome}:{start}-{stop}"
    )

In [7]:
def calculate_af(ac, an):
    if ac is None or an in [None, 0]:
        return None

    try:
        return float(ac) / float(an)
    except (TypeError, ValueError, ZeroDivisionError):
        return None


def get_af(block):
    if not isinstance(block, dict):
        return None

    af = block.get("af")

    if af is not None:
        return af

    return calculate_af(block.get("ac"), block.get("an"))


def variant_type_from_alleles(ref, alt):
    if pd.isna(ref) or pd.isna(alt):
        return None

    ref = str(ref)
    alt = str(alt)

    if len(ref) == 1 and len(alt) == 1:
        return "SNV"

    return "indel"

In [8]:
def get_max_population_frequency(exome, genome):
    max_info = {
        "MAX_freq": None,
        "MAX_freq_source": None,
        "MAX_freq_population": None,
        "MAX_freq_AC": None,
        "MAX_freq_AN": None,
    }

    for source_name, block in {
        "exome": exome,
        "genome": genome,
    }.items():
        populations = block.get("populations") or []

        for population in populations:
            population_id = population.get("id")
            ac = population.get("ac")
            an = population.get("an")
            af = get_af(population)

            if af is None:
                continue

            if max_info["MAX_freq"] is None or af > max_info["MAX_freq"]:
                max_info = {
                    "MAX_freq": af,
                    "MAX_freq_source": source_name,
                    "MAX_freq_population": population_id,
                    "MAX_freq_AC": ac,
                    "MAX_freq_AN": an,
                }

    return max_info

In [9]:
def flatten_gnomad_variant(variant, gene_info):
    exome = variant.get("exome") or {}
    genome = variant.get("genome") or {}

    rsids = variant.get("rsids") or []
    rsid = ";".join(rsids) if rsids else None

    ref = variant.get("ref")
    alt = variant.get("alt")

    max_population_info = get_max_population_frequency(
        exome=exome,
        genome=genome
    )

    row = {
        "gene_symbol": gene_info["gene_symbol"],
        "official_gene_symbol": gene_info["official_gene_symbol"],
        "variant_id": variant.get("variant_id"),
        "rsID": rsid,
        "Variant_type": variant_type_from_alleles(ref, alt),
        "Alleles_REF_ALT": f"{ref}>{alt}" if ref and alt else None,
        "chromosome": variant.get("chrom"),
        "position_GRCh38": variant.get("pos"),
        "reference_genome": reference_genome,
        "Functional_consequence": variant.get("consequence"),
        "gnomAD_exome_AC": exome.get("ac"),
        "gnomAD_exome_AN": exome.get("an"),
        "gnomAD_exome_AF": get_af(exome),
        "gnomAD_genome_AC": genome.get("ac"),
        "gnomAD_genome_AN": genome.get("an"),
        "gnomAD_genome_AF": get_af(genome),
        "MAX_freq": max_population_info["MAX_freq"],
        "MAX_freq_source": max_population_info["MAX_freq_source"],
        "MAX_freq_population": max_population_info["MAX_freq_population"],
        "MAX_freq_AC": max_population_info["MAX_freq_AC"],
        "MAX_freq_AN": max_population_info["MAX_freq_AN"],
    }

    return row

In [10]:
def collect_gnomad_rows_for_gene(gene_region, chunk_size=25_000, request_pause=1):
    gene_symbol = gene_region["input_gene_symbol"]
    official_gene_symbol = gene_region["official_gene_symbol"]

    chromosome = str(gene_region["chromosome"]).replace("chr", "")

    gene_start = int(gene_region["gene_start"])
    gene_end = int(gene_region["gene_end"])

    # For this complementary gnomAD frequency table, query only the gene body.
    query_start = gene_start
    query_end = gene_end

    region_chunks = make_region_chunks(
        start=query_start,
        end=query_end,
        chunk_size=chunk_size
    )

    all_variants = []

    for start, stop in region_chunks:
        variants = query_gnomad_region(
            chromosome=chromosome,
            start=start,
            stop=stop,
            dataset=dataset,
        )

        all_variants.extend(variants)

        time.sleep(request_pause)

    # Remove duplicate variants within this gene.
    # This keeps the same variant once per gene.
    variant_dict = {}

    for variant in all_variants:
        variant_id = variant.get("variant_id")

        if variant_id:
            variant_dict[variant_id] = variant

    unique_variants = list(variant_dict.values())

    gene_info = {
        "gene_symbol": gene_symbol,
        "official_gene_symbol": official_gene_symbol,
        "chromosome": chromosome,
        "query_start": query_start,
        "query_end": query_end,
    }

    rows = [
        flatten_gnomad_variant(variant, gene_info)
        for variant in unique_variants
    ]

    summary_row = {
        "gene_symbol": gene_symbol,
        "official_gene_symbol": official_gene_symbol,
        "chromosome": chromosome,
        "query_region": f"chr{chromosome}:{query_start}-{query_end}",
        "variants_retrieved": len(unique_variants),
        "chunks_queried": len(region_chunks),
    }

    return rows, summary_row

In [11]:
all_rows = []
gene_query_summaries = []

GENE_PAUSE = 3       # seconds between genes (on top of per-chunk pauses)
CHUNK_PAUSE = 1.5    # slightly more breathing room per chunk

# Resume support: skip genes already saved
already_done = set()
if out_csv.exists():
    existing = pd.read_csv(out_csv, usecols=["gene_symbol"], sep=";")
    already_done = set(existing["gene_symbol"].str.upper())
    print(f"Resuming — {len(already_done)} genes already in {out_csv}")

for _, gene_region in gene_regions.iterrows():
    gene_symbol = gene_region["input_gene_symbol"]

    if gene_symbol.upper() in already_done:
        print(f"Skipping {gene_symbol} (already saved)")
        continue

    print(f"Querying gnomAD for {gene_symbol}...")

    try:
        rows, summary_row = collect_gnomad_rows_for_gene(
            gene_region=gene_region,
            chunk_size=25_000,
            request_pause=CHUNK_PAUSE,
        )
    except RuntimeError as e:
        print(f"  ERROR for {gene_symbol}: {e}. Skipping.")
        gene_query_summaries.append({
            "gene_symbol": gene_symbol,
            "official_gene_symbol": gene_region["official_gene_symbol"],
            "chromosome": gene_region["chromosome"],
            "query_region": None,
            "variants_retrieved": 0,
            "chunks_queried": None,
            "error": str(e),
        })
        time.sleep(GENE_PAUSE)
        continue

    all_rows.extend(rows)
    gene_query_summaries.append(summary_row)

    # Incrementally append to CSV so a crash doesn't lose prior work
    if rows:
        gene_df = pd.DataFrame(rows)
        write_header = not out_csv.exists()
        gene_df.to_csv(out_csv, mode="a", index=False, header=write_header, sep=";")
        print(f"{len(rows)} variants saved for {gene_symbol}")

    time.sleep(GENE_PAUSE)

# Build final df from saved file (handles both fresh and resumed runs)
df = pd.read_csv(out_csv, sep=";") if out_csv.exists() else pd.DataFrame(all_rows)
gene_query_summary_df = pd.DataFrame(gene_query_summaries)

df.shape

Querying gnomAD for APOE...
5220 variants saved for APOE
Querying gnomAD for B2M...
5513 variants saved for B2M
Querying gnomAD for C1QA...
3145 variants saved for C1QA
Querying gnomAD for C1QB...
4412 variants saved for C1QB
Querying gnomAD for C1QC...
3442 variants saved for C1QC
Querying gnomAD for C1R...
8364 variants saved for C1R
Querying gnomAD for C2...
21727 variants saved for C2
Querying gnomAD for CAPG...
17463 variants saved for CAPG
Querying gnomAD for CCL2...
1977 variants saved for CCL2
Querying gnomAD for CCL3...
1988 variants saved for CCL3
Querying gnomAD for CCL7...
1473 variants saved for CCL7
Querying gnomAD for CD14...
2278 variants saved for CD14
Querying gnomAD for CD163...
13453 variants saved for CD163
Querying gnomAD for CFD...
12142 variants saved for CFD
Querying gnomAD for CFH...
37129 variants saved for CFH
Querying gnomAD for CSF1...
12033 variants saved for CSF1
Querying gnomAD for CSF1R...
26786 variants saved for CSF1R
Querying gnomAD for CST3...
7723

/var/folders/ml/rhzbx8mx0jj25ftqqhfslrkc0000gn/T/ipykernel_63011/1683999365.py:56: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(out_csv, sep=";") if out_csv.exists() else pd.DataFrame(all_rows)


(840229, 21)

In [12]:
df["MAX_freq"] = pd.to_numeric(df["MAX_freq"], errors="coerce")

df["Priority"] = df["MAX_freq"].apply(
    lambda x: "Unknown" if pd.isna(x)
    else "Primary" if x > 0.001
    else "Secondary" if 0.0001 < x <= 0.001
    else "Below threshold"
)

df[["variant_id", "rsID", "MAX_freq", "MAX_freq_source", "MAX_freq_population", "Priority"]].head()

,variant_id,rsID,MAX_freq,MAX_freq_source,MAX_freq_population,Priority
0,19-44903787-G-A,NaN,0.0,exome,remaining,Below threshold
1,19-44903788-C-T,rs1969733077,0.0,exome,remaining,Below threshold
2,19-44903789-C-T,rs1171628249,0.0,exome,remaining,Below threshold
3,19-44903790-A-G,NaN,0.0,exome,remaining,Below threshold
4,19-44903790-A-T,NaN,0.0,exome,remaining,Below threshold


In [13]:
df = (
    df
    .sort_values("MAX_freq", ascending=False, na_position="last")
    .reset_index(drop=True)
)

df[["gene_symbol", "variant_id", "rsID", "MAX_freq", "Priority"]].head()

,gene_symbol,variant_id,rsID,MAX_freq,Priority
0,MSR1,8-16309070-C-T,rs1366961,1.0,Primary
1,MSR1,8-16355879-C-A,rs1896911,1.0,Primary
2,MSR1,8-16322153-T-G,rs1366955,1.0,Primary
3,CAPG,2-85421752-C-G,rs11678715,1.0,Primary
4,CSF1R,5-150067043-C-A,rs12521625,1.0,Primary


In [14]:
final_columns = [
    "gene_symbol",
    "official_gene_symbol",
    "variant_id",
    "rsID",
    "Variant_type",
    "Alleles_REF_ALT",
    "chromosome",
    "position_GRCh38",
    "reference_genome",
    "Functional_consequence",
    "gnomAD_exome_AC",
    "gnomAD_exome_AN",
    "gnomAD_exome_AF",
    "gnomAD_genome_AC",
    "gnomAD_genome_AN",
    "gnomAD_genome_AF",
    "MAX_freq",
    "MAX_freq_source",
    "MAX_freq_population",
    "MAX_freq_AC",
    "MAX_freq_AN",
    "Priority",
]

df = df[final_columns]

df.head()

,gene_symbol,official_gene_symbol,variant_id,rsID,Variant_type,Alleles_REF_ALT,chromosome,position_GRCh38,reference_genome,Functional_consequence,...,gnomAD_exome_AF,gnomAD_genome_AC,gnomAD_genome_AN,gnomAD_genome_AF,MAX_freq,MAX_freq_source,MAX_freq_population,MAX_freq_AC,MAX_freq_AN,Priority
0,MSR1,MSR1,8-16309070-C-T,rs1366961,SNV,C>T,8,16309070,GRCh38,intron_variant,...,NaN,148383.0,152154.0,0.975216,1.0,genome,fin,10612.0,10612.0,Primary
1,MSR1,MSR1,8-16355879-C-A,rs1896911,SNV,C>A,8,16355879,GRCh38,intron_variant,...,NaN,140849.0,152110.0,0.925968,1.0,genome,ami,900.0,900.0,Primary
2,MSR1,MSR1,8-16322153-T-G,rs1366955,SNV,T>G,8,16322153,GRCh38,intron_variant,...,NaN,148122.0,152152.0,0.973513,1.0,genome,fin,10620.0,10620.0,Primary
3,CAPG,CAPG,2-85421752-C-G,rs11678715,SNV,C>G,2,85421752,GRCh38,intron_variant,...,0.336000,35936.0,151938.0,0.236518,1.0,exome,asj,2.0,2.0,Primary
4,CSF1R,CSF1R,5-150067043-C-A,rs12521625,SNV,C>A,5,150067043,GRCh38,intron_variant,...,0.514963,64700.0,152014.0,0.425619,1.0,exome,mid,2.0,2.0,Primary


In [15]:
df = df.drop_duplicates(subset=["gene_symbol", "variant_id"])

df.to_csv(out_csv, index=False, sep=";")

In [16]:
summary = {
    "total_rows": int(len(df)),
    "unique_genes_in_output": int(df["gene_symbol"].nunique(dropna=True)),
    "unique_variant_ids": int(df["variant_id"].nunique(dropna=True)),
    "unique_rsIDs": int(df["rsID"].nunique(dropna=True)),
    "priority_counts": df["Priority"].value_counts(dropna=False).to_dict(),
}

summary

{'total_rows': 840229,
 'unique_genes_in_output': 52,
 'unique_variant_ids': 840229,
 'unique_rsIDs': 480378,
 'priority_counts': {'Below threshold': 628886,
  'Secondary': 148643,
  'Primary': 60685,
  'Unknown': 2015}}